# 🔴 Audit « équipe rouge » du modèle de scores (MODEL_POISSON)

**Groupe :** Maxime, Loan, Moussa, Mouhamedou et Ousmane — audit : Loan

## Le rôle de ce notebook
Le but n'est pas d'être gentil avec le modèle : **c'est d'essayer de le casser**. On cherche une fuite d'information, une mesure trompeuse ou un résultat qui ne tient que sur un petit échantillon. Si le modèle résiste, on peut le défendre. S'il casse, il vaut mieux le savoir avant la soutenance.

**Le modèle audité** (`MODEL_POISSON.ipynb`, Moussa) : chaque équipe marque un nombre de buts qui suit une loi de Poisson, avec log λ = constante + β₁ × écart d'Elo complet / 100 + β₂ × terrain (+ β₃ × nouveau format). Validation glissante 2015-2022, test 2023-2025.

## Règle de cet audit
- **On ne modifie pas le modèle.** On le recode à l'identique, on vérifie qu'on retrouve ses chiffres, puis on le soumet à des tests.
- **Attention au test 2023-2025 :** l'audit l'utilise plusieurs fois (sous-groupes, variables retirées). C'est permis pour un **diagnostic**, mais **aucun choix de modèle ne doit être pris à partir de ces chiffres**, sinon le test ne serait plus un test.

## Plan
1. Reproduire les chiffres de MODEL_POISSON
2. Test 1 : la cible mélangée
3. Test 2 : l'Elo officiel est-il vraiment connu avant le match ?
4. Test 3 : retirer les variables (terrain, nouveau format)
5. Test 4 : les scores par sous-groupe sur le test
6. Test 5 : la fuite de l'Elo « maison recalé »
7. Audit du test de la forme récente
8. Conclusion : ce qui résiste, ce qui casse, ce qu'on ne peut pas affirmer

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from math import factorial

%config InlineBackend.figure_format = 'retina'
BLEU, ORANGE, GRIS, ENCRE = '#2a78d6', '#eb6834', '#b5b3ad', '#52514e'
ISSUES = ['Victoire domicile', 'Nul', 'Victoire extérieur']
MAX_BUTS = 10
GRAINE = 42

g = pd.read_csv('../output/eda_matchs_c1.csv', encoding='utf-8-sig', parse_dates=['date'])
g = g.rename(columns={'dom_j_buts_pour': 'buts_dom', 'ext_j_buts_pour': 'buts_ext'})
g['neutre'] = g['neutre'].astype(bool)
g['domicile'] = (~g['neutre']).astype(int)
g['nouveau_format'] = (g['season'] >= 2024).astype(int)

# Phase du match, depuis games.csv : groupes / phase de ligue, ou élimination directe
rounds = pd.read_csv('../data/games.csv', usecols=['game_id', 'round'])
g = g.merge(rounds, on='game_id', how='left', validate='one_to_one')
g['phase'] = np.where(g['round'].str.startswith('Group'), 'groupes / phase de ligue', 'élimination directe')

# Forme récente (v2_forme.ipynb) et couverture du championnat (v2_fatigue.ipynb)
forme = pd.read_csv('../output/forme_par_match.csv', encoding='utf-8-sig')
g = g.merge(forme[['game_id', 'ecart_forme_5']], on='game_id', how='left', validate='one_to_one')
g['deux_couverts'] = g['dom_f_couverture_championnat_detectee'].astype(bool) & g['ext_f_couverture_championnat_detectee'].astype(bool)

assert len(g) == 1871 and g['game_id'].is_unique
assert g[['buts_dom', 'buts_ext', 'ecart_elo_complet', 'round']].notna().all().all()
print(f"{len(g):,} matchs | {g['phase'].value_counts().to_dict()}")

1,871 matchs | {'groupes / phase de ligue': 1440, 'élimination directe': 431}


## 1. Reproduire les chiffres de MODEL_POISSON

**Pourquoi :** avant de critiquer un modèle, il faut être sûr d'auditer **le même modèle**. On recode les fonctions de Moussa (même formule, mêmes données, même découpage) en les généralisant pour pouvoir ajouter ou retirer une variable, et on vérifie qu'on retrouve ses chiffres.

Chiffres à retrouver : validation glissante (Poisson Elo + terrain) **log loss 0,9148**, Dummy 1,0536 ; test 2023-2025 : Poisson **0,902**, Dummy **1,020**, accuracy **58,6 %**.

In [2]:
# Chaque variable : comment la lire dans un match, et son signe vu de l'équipe qui se déplace
SOURCES = {
    'ecart_100': (lambda d: d['ecart_elo_complet'] / 100, -1),
    'terrain': (lambda d: d['domicile'], -1),
    'nouveau_format': (lambda d: d['nouveau_format'], +1),   # même effet pour les deux équipes
    'ecart_forme': (lambda d: d['ecart_forme_5'].fillna(0), -1),
}

def lignes_equipes(d, variables, buts_dom=None, buts_ext=None):
    # Chaque match devient 2 lignes : une par équipe, vue de son côté
    buts_dom = d['buts_dom'].to_numpy() if buts_dom is None else buts_dom
    buts_ext = d['buts_ext'].to_numpy() if buts_ext is None else buts_ext
    t = {'buts': np.concatenate([buts_dom, buts_ext])}
    for v in variables:
        f, signe = SOURCES[v]
        x = f(d).to_numpy(dtype=float)
        t[v] = np.concatenate([x, signe * x])
    return pd.DataFrame(t)

def entrainer(d, variables, **buts):
    lignes = lignes_equipes(d, variables, **buts)
    return sm.GLM(lignes['buts'], sm.add_constant(lignes[variables], has_constant='add'),
                  family=sm.families.Poisson()).fit().params

def buts_attendus_matchs(coefs, d, variables):
    lin_dom = np.full(len(d), coefs['const'])
    lin_ext = np.full(len(d), coefs['const'])
    for v in variables:
        f, signe = SOURCES[v]
        x = f(d).to_numpy(dtype=float)
        lin_dom = lin_dom + coefs[v] * x
        lin_ext = lin_ext + coefs[v] * signe * x
    return np.exp(lin_dom), np.exp(lin_ext)

K = np.arange(MAX_BUTS + 1)
FACT = np.array([factorial(k) for k in K], dtype=float)
DOM_GAGNE = np.tril(np.ones((MAX_BUTS + 1, MAX_BUTS + 1)), -1).astype(bool)
NUL = np.eye(MAX_BUTS + 1, dtype=bool)

def probas_issues(lam_dom, lam_ext):
    p_dom = np.exp(-lam_dom[:, None]) * lam_dom[:, None] ** K / FACT
    p_ext = np.exp(-lam_ext[:, None]) * lam_ext[:, None] ** K / FACT
    m = p_dom[:, :, None] * p_ext[:, None, :]
    m = m / m.sum(axis=(1, 2), keepdims=True)
    return pd.DataFrame({'Victoire domicile': m[:, DOM_GAGNE].sum(axis=1), 'Nul': m[:, NUL].sum(axis=1),
                         'Victoire extérieur': m[:, ~DOM_GAGNE & ~NUL].sum(axis=1)})[ISSUES]

def predire(appr, a_predire, variables, **buts):
    coefs = entrainer(appr, variables, **buts)
    return probas_issues(*buts_attendus_matchs(coefs, a_predire, variables))

def predire_dummy(appr, a_predire):
    freq = appr['resultat'].value_counts(normalize=True)[ISSUES]
    return pd.DataFrame([freq.to_numpy()] * len(a_predire), columns=ISSUES)

def perte(y, probas):
    # Log loss match par match
    vrai = pd.Series(y).map({issue: i for i, issue in enumerate(ISSUES)}).to_numpy()
    p = probas[ISSUES].to_numpy()
    return -np.log(np.clip(p[np.arange(len(p)), vrai], 1e-15, 1))

def mesures(y, probas):
    vrai = pd.Series(y).map({issue: i for i, issue in enumerate(ISSUES)}).to_numpy()
    p = probas[ISSUES].to_numpy()
    observe = np.eye(3)[vrai]
    rps = ((np.cumsum(p, axis=1)[:, :2] - np.cumsum(observe, axis=1)[:, :2]) ** 2).sum(axis=1) / 2
    return {'log loss': perte(y, probas).mean(), 'RPS': rps.mean(), 'accuracy': (p.argmax(axis=1) == vrai).mean()}

def ic_ecart(perte_a, perte_b, n_tirages=5000):
    # Bootstrap apparié : intervalle à 95 % de l'écart moyen de log loss (a − b)
    rng = np.random.default_rng(GRAINE)
    ecart = perte_a - perte_b
    tirages = ecart[rng.integers(0, len(ecart), size=(n_tirages, len(ecart)))].mean(axis=1)
    return ecart.mean(), *np.percentile(tirages, [2.5, 97.5])

VARIABLES = ['ecart_100', 'terrain']
SAISONS_VALIDATION = range(2015, 2023)

def glissante(variables, donnees=g, evaluer_sur=None):
    # Validation glissante : pour chaque saison, apprentissage sur toutes les saisons d'avant
    evaluer_sur = donnees if evaluer_sur is None else evaluer_sur
    lignes = []
    for s in SAISONS_VALIDATION:
        appr, val = donnees[donnees['season'] < s], evaluer_sur[evaluer_sur['season'] == s]
        if val.empty:                                  # saison sans match à évaluer (sous-groupe)
            continue
        lignes.append({'saison': s, 'matchs': len(val), 'log loss': perte(val['resultat'], predire(appr, val, variables)).mean(),
                       'log loss Dummy': perte(val['resultat'], predire_dummy(appr, val)).mean()})
    return pd.DataFrame(lignes).set_index('saison')

def moyenne_ponderee(t, colonne):
    return np.average(t[colonne], weights=t['matchs'])

val_ref = glissante(VARIABLES)
appr_test, test = g[g['season'] <= 2022], g[g['season'] >= 2023]
probas_ref = predire(appr_test, test, VARIABLES)
perte_ref = perte(test['resultat'], probas_ref)
reference = pd.DataFrame({'Poisson (Elo + terrain)': mesures(test['resultat'], probas_ref),
                          'Dummy': mesures(test['resultat'], predire_dummy(appr_test, test))}).T

print(f"Validation glissante : Poisson {moyenne_ponderee(val_ref, 'log loss'):.4f} | Dummy {moyenne_ponderee(val_ref, 'log loss Dummy'):.4f}")
print(f"Test 2023-2025 ({len(test)} matchs) :")
print(reference.round(4).to_string())

assert round(moyenne_ponderee(val_ref, 'log loss'), 4) == 0.9148
assert round(reference.loc['Poisson (Elo + terrain)', 'log loss'], 3) == 0.902
assert round(reference.loc['Dummy', 'log loss'], 3) == 1.020
print("\nChiffres de MODEL_POISSON retrouvés : on audite bien le même modèle.")

Validation glissante : Poisson 0.9148 | Dummy 1.0536
Test 2023-2025 (502 matchs) :
                         log loss     RPS  accuracy
Poisson (Elo + terrain)    0.9023  0.1932    0.5857
Dummy                      1.0196  0.2359    0.5000

Chiffres de MODEL_POISSON retrouvés : on audite bien le même modèle.


**Lecture :** on retrouve **exactement** les chiffres de `MODEL_POISSON` : log loss de **0,9148** en validation glissante (Dummy 1,0536), et sur le test **0,902** (Dummy 1,020), avec un RPS de 0,1932 et une accuracy de **58,6 %** (Dummy 50,0 %). Les `assert` le vérifient. On audite donc bien le même modèle, et tous les tests qui suivent partent de cette référence.

## 2. Test 1 : la cible mélangée

**L'idée :** on **mélange au hasard les scores** des matchs d'apprentissage : chaque match reçoit le score d'un autre match. Le lien entre l'Elo et les buts est alors détruit. Un modèle honnête ne peut plus rien apprendre et doit tomber **au niveau du Dummy, ou pire**. S'il restait bon sur le test, c'est qu'une information sur le résultat passerait par un autre chemin : une fuite.

On le fait **20 fois** (graines fixes) pour voir la variation due au hasard.

In [3]:
rng = np.random.default_rng(GRAINE)
melanges = []
for k in range(20):
    ordre = rng.permutation(len(appr_test))                       # on mélange les couples (buts dom, buts ext)
    buts = dict(buts_dom=appr_test['buts_dom'].to_numpy()[ordre], buts_ext=appr_test['buts_ext'].to_numpy()[ordre])
    coefs = entrainer(appr_test, VARIABLES, **buts)
    p = probas_issues(*buts_attendus_matchs(coefs, test, VARIABLES))
    melanges.append({'tirage': k, 'log loss test': perte(test['resultat'], p).mean(),
                     'coef Elo': coefs['ecart_100'], 'coef terrain': coefs['terrain']})
melanges = pd.DataFrame(melanges)

coefs_vrais = entrainer(appr_test, VARIABLES)
# Le mélange garde les buts de l'équipe qui reçoit de son côté : l'avantage du terrain survit.
# La bonne référence est donc le modèle « terrain seul », qui ne sait rien des deux équipes.
terrain_seul = perte(test['resultat'], predire(appr_test, test, ['terrain'])).mean()
resume = pd.DataFrame({
    'log loss test': [reference.loc['Poisson (Elo + terrain)', 'log loss'], melanges['log loss test'].mean(),
                      melanges['log loss test'].min(), terrain_seul, reference.loc['Dummy', 'log loss']],
    'coef Elo': [coefs_vrais['ecart_100'], melanges['coef Elo'].mean(), melanges['coef Elo'].abs().max(), np.nan, np.nan],
    'coef terrain': [coefs_vrais['terrain'], melanges['coef terrain'].mean(), np.nan, np.nan, np.nan],
}, index=['vrais scores', 'scores mélangés (moyenne de 20)', 'scores mélangés (meilleur tirage)',
          'terrain seul (sans Elo)', 'Dummy'])
print(resume.round(4).to_string())
assert melanges['log loss test'].min() > reference.loc['Poisson (Elo + terrain)', 'log loss'] + 0.05, "Le modèle mélangé reste bon : fuite possible"
assert abs(melanges['log loss test'].mean() - terrain_seul) < 0.02, "Le modèle mélangé fait mieux que « terrain seul »"

                                   log loss test  coef Elo  coef terrain
vrais scores                              0.9023    0.2031        0.1233
scores mélangés (moyenne de 20)           1.0260    0.0020        0.1183
scores mélangés (meilleur tirage)         1.0078    0.0170           NaN
terrain seul (sans Elo)                   1.0281       NaN           NaN
Dummy                                     1.0196       NaN           NaN


**Lecture :**
- **Avec des scores mélangés, le modèle s'effondre** : log loss de **1,026** en moyenne sur 20 tirages (1,008 au mieux), contre **0,902** avec les vrais scores. Le coefficient de l'Elo tombe de **0,203 à 0,002** : sans le vrai lien entre l'Elo et les buts, le modèle n'apprend plus rien de l'Elo.
- **Le coefficient du terrain, lui, reste (0,118 contre 0,123)** : le mélange déplace des scores entiers, donc les buts de l'équipe qui reçoit restent de son côté. Le modèle mélangé devient un modèle « terrain seul », et il en a bien le niveau (**1,026** contre **1,028**). C'est pour ça qu'il frôle le Dummy (1,020) : les deux ignorent tout des équipes.
- **Correction de notre propre test :** notre premier contrôle exigeait que le modèle mélangé ne batte jamais le Dummy, et il a échoué (1,008 < 1,020). Ce n'était pas une fuite, mais une erreur de référence de notre part : la bonne référence est « terrain seul ». On l'a corrigée, et on le signale.
- **Conclusion :** aucune information sur le résultat ne passe par un autre chemin que les variables. **Mais ce test ne voit pas une fuite cachée DANS une variable** (un Elo calculé après le match) : c'est l'objet du test suivant.

## 3. Test 2 : l'Elo officiel est-il vraiment connu avant le match ?

**Pourquoi ce test :** le test 1 ne peut pas détecter une fuite **dans les variables**. Si l'écart d'Elo contenait déjà le résultat du match (un Elo calculé *après* le match), le mélange des scores casserait aussi le lien, et le test 1 passerait quand même. Il faut donc vérifier directement la variable la plus importante.

**Le test :** 89 % des Elo complets sont des Elo **officiels** (`EloRatings.csv`). Pour chacun, on cherche dans `EloRatings.csv` un relevé qui a **exactement la même valeur**. On regarde si ce relevé est daté **avant** le match (normal) ou seulement le jour même ou après (fuite).

In [4]:
elo_officiel = pd.read_csv('../data/EloRatings.csv', parse_dates=['date'])
elo_officiel['valeur'] = elo_officiel['elo'].round(2)
releves = elo_officiel.groupby('valeur')['date'].agg(['min', 'max'])

lignes = []
for cote in ['home', 'away']:
    t = g.loc[g[f'{cote}_source_elo'] == 'officiel', ['game_id', 'date', f'{cote}_elo_complet']].copy()
    t['valeur'] = t[f'{cote}_elo_complet'].round(2)
    lignes.append(t[['game_id', 'date', 'valeur']])
officiels = pd.concat(lignes)

# Pour chaque Elo officiel utilisé : existe-t-il un relevé de même valeur daté avant le match ?
avant = officiels.merge(elo_officiel[['valeur', 'date']].rename(columns={'date': 'date_releve'}), on='valeur')
avant = avant[avant['date_releve'] < avant['date']]
trouve = officiels.set_index(['game_id', 'valeur']).index.isin(avant.set_index(['game_id', 'valeur']).index)
dernier_avant = avant.groupby(['game_id', 'valeur'])['date_releve'].max()
anciennete = (officiels.set_index(['game_id', 'valeur'])['date'] - dernier_avant.reindex(officiels.set_index(['game_id', 'valeur']).index)).dt.days

print(f"Elo officiels utilisés (lignes club-match) : {len(officiels):,}")
print(f"Retrouvés dans EloRatings.csv à une date AVANT le match : {trouve.mean():.1%}")
print(f"Ancienneté du relevé utilisé : médiane {anciennete.median():.0f} jours, maximum {anciennete.max():.0f} jours")
assert trouve.all(), "Un Elo officiel n'existe qu'à une date postérieure au match"

Elo officiels utilisés (lignes club-match) : 3,348
Retrouvés dans EloRatings.csv à une date AVANT le match : 100.0%
Ancienneté du relevé utilisé : médiane 7 jours, maximum 17 jours


**Lecture :**
- **Les 3 348 Elo officiels utilisés (89 % des lignes) existent tous dans `EloRatings.csv` à une date antérieure au match.** Le relevé utilisé date en général de **7 jours** avant le match, et jamais de plus de **17 jours**. C'est cohérent avec deux relevés par mois.
- **Limite du test :** on retrouve les relevés par leur **valeur**, pas par le nom du club. Deux clubs pourraient avoir exactement le même Elo, au centième près. Mais l'ancienneté, toujours de 17 jours ou moins, colle trop bien au rythme des relevés pour être due au hasard.
- **Les 11 % d'Elo estimés** ne peuvent pas être vérifiés ainsi. On a relu le code de `v2_elo.ipynb` : l'estimation « par les résultats » n'utilise que des matchs **strictement avant** la date (`t['date'] < date`). Le recalage, lui, pose problème (test 5).

## 4. Test 3 : retirer les variables

**L'idée :** on retire une variable et on mesure **de combien la log loss se dégrade**. Si elle ne se dégrade pas, la variable ne sert à rien. Si elle se dégrade beaucoup, on vérifie que ce n'est pas trop beau pour être vrai. On donne à chaque fois l'**intervalle à 95 %** (bootstrap apparié, 5 000 tirages) : si l'intervalle contient 0, on ne peut pas dire que la variable aide.

- **Terrain** et **Elo** : mesurés sur la validation glissante et sur le test ;
- **Nouveau format** : il n'existe que depuis 2024. Il ne peut donc être appris qu'avec 2024 dans l'apprentissage, et jugé que sur **2025-2026 (188 matchs)**, comme l'a fait Moussa.

In [5]:
def comparer(variables_a, variables_b, appr, evalue):
    # Écart de log loss (a − b) et intervalle à 95 % : positif = a est moins bon que b
    pa = perte(evalue['resultat'], predire(appr, evalue, variables_a))
    pb = perte(evalue['resultat'], predire(appr, evalue, variables_b))
    return ic_ecart(pa, pb)

lignes = []
for nom, variables in [('sans le terrain (Elo seul)', ['ecart_100']), ("sans l'Elo (terrain seul)", ['terrain'])]:
    v = glissante(variables)
    ecart, bas, haut = comparer(variables, VARIABLES, appr_test, test)
    lignes.append({'modèle': nom, 'log loss glissante': moyenne_ponderee(v, 'log loss'),
                   'écart glissante': moyenne_ponderee(v, 'log loss') - moyenne_ponderee(val_ref, 'log loss'),
                   'écart test': ecart, 'IC 95 % bas': bas, 'IC 95 % haut': haut})
retrait = pd.DataFrame(lignes).set_index('modèle')
print("Écart de log loss par rapport au modèle complet (positif = le modèle se dégrade) :")
print(retrait.round(4).to_string())

# Nouveau format : apprentissage 2012-2024, jugé sur 2025-2026
appr_25, saison_25 = g[g['season'] <= 2024], g[g['season'] == 2025]
ecart_f, bas_f, haut_f = comparer(VARIABLES, VARIABLES + ['nouveau_format'], appr_25, saison_25)
print(f"\nRetirer le nouveau format (jugé sur 2025-2026, {len(saison_25)} matchs) : écart {ecart_f:+.4f} "
      f"(IC 95 % : {bas_f:+.4f} à {haut_f:+.4f})")

Écart de log loss par rapport au modèle complet (positif = le modèle se dégrade) :
                            log loss glissante  écart glissante  écart test  IC 95 % bas  IC 95 % haut
modèle                                                                                                
sans le terrain (Elo seul)              0.9307           0.0159      0.0224       0.0072        0.0376
sans l'Elo (terrain seul)               1.0558           0.1410      0.1258       0.0853        0.1650

Retirer le nouveau format (jugé sur 2025-2026, 188 matchs) : écart +0.0016 (IC 95 % : -0.0037 à +0.0066)


**Lecture :**

| Variable retirée | Écart validation | Écart test | Intervalle à 95 % (test) | Verdict |
|---|---|---|---|---|
| Terrain | +0,016 | +0,022 | +0,007 à +0,038 | **utile**, effet réel |
| Elo | +0,141 | +0,126 | +0,085 à +0,165 | **indispensable** |
| Nouveau format (jugé sur 2025-2026) | — | +0,0016 | **−0,0037 à +0,0066** | **pas démontré** |

- **L'Elo fait presque tout le travail** : sans lui, le modèle retombe au niveau du Dummy (1,056 en validation).
- **Le terrain apporte un vrai gain**, petit mais net : l'intervalle ne contient pas 0.
- **Le nouveau format ne résiste pas** : l'amélioration annoncée dans `MODEL_POISSON` (« améliore tout », 0,9320 → 0,9304) est de **0,0016**, et l'intervalle **contient 0**. Sur 188 matchs, **on ne peut pas affirmer** que cette variable améliore la prédiction de 2025-2026.
- **Un second problème, de méthode :** la saison 2025-2026 fait **partie du test** (2023-2025). C'est en regardant le test (section 6, buts sous-estimés) que l'idée d'ajouter le format est venue, et c'est encore sur le test qu'elle a été validée. Ce n'est pas une fuite de données, mais **le test n'est plus vierge** pour cette variable.
- **Ce qui reste défendable :** la variable est connue avant le match, et l'idée est raisonnable (+14 % de buts observés depuis 2024). Mais il faut la présenter comme une **hypothèse plausible, non démontrée**, et pas comme un gain prouvé.

## 5. Test 4 : les scores par sous-groupe sur le test

**Pourquoi :** une moyenne peut cacher un modèle qui marche bien sur les matchs faciles et mal ailleurs. On découpe les 502 matchs de test en sous-groupes, et pour chacun on compare le Poisson au Dummy :
- **phase** : groupes / phase de ligue contre élimination directe ;
- **écart d'Elo** : petit (moins de 100 points) contre grand (100 points ou plus). Le seuil de 100 est fixé à l'avance, c'est l'unité du modèle ;
- **origine de l'Elo** : les deux Elo officiels, contre au moins un Elo estimé.

On donne aussi l'intervalle à 95 % du gain du Poisson sur le Dummy : sur un petit sous-groupe, il est large.

In [6]:
perte_dummy = perte(test['resultat'], predire_dummy(appr_test, test))
t = test.assign(perte_poisson=perte_ref, perte_dummy=perte_dummy,
                bon=(probas_ref.to_numpy().argmax(axis=1) == test['resultat'].map({i: k for k, i in enumerate(ISSUES)}).to_numpy()))
decoupages = {
    'phase': t['phase'],
    "écart d'Elo": np.where(t['ecart_elo_complet'].abs() < 100, 'petit (< 100 points)', 'grand (≥ 100 points)'),
    "origine de l'Elo": np.where((t['home_source_elo'] == 'officiel') & (t['away_source_elo'] == 'officiel'),
                                 'deux Elo officiels', 'au moins un Elo estimé'),
}
lignes = []
for decoupage, groupes in decoupages.items():
    for groupe in pd.unique(groupes):
        d = t[groupes == groupe]
        gain, bas, haut = ic_ecart(d['perte_dummy'].to_numpy(), d['perte_poisson'].to_numpy())
        lignes.append({'découpage': decoupage, 'groupe': groupe, 'matchs': len(d),
                       'log loss Poisson': d['perte_poisson'].mean(), 'log loss Dummy': d['perte_dummy'].mean(),
                       'gain sur le Dummy': gain, 'IC 95 % bas': bas, 'IC 95 % haut': haut, 'accuracy': d['bon'].mean()})
sous_groupes = pd.DataFrame(lignes).set_index(['découpage', 'groupe'])
assert sous_groupes.groupby(level=0)['matchs'].sum().eq(len(test)).all()
print(sous_groupes.round(3).to_string())

                                           matchs  log loss Poisson  log loss Dummy  gain sur le Dummy  IC 95 % bas  IC 95 % haut  accuracy
découpage        groupe                                                                                                                    
phase            groupes / phase de ligue     384             0.873           1.016              0.143        0.099         0.185     0.604
                 élimination directe          118             0.999           1.033              0.034       -0.043         0.106     0.525
écart d'Elo      grand (≥ 100 points)         287             0.809           0.996              0.187        0.122         0.248     0.672
                 petit (< 100 points)         215             1.026           1.050              0.024        0.002         0.046     0.470
origine de l'Elo au moins un Elo estimé        93             0.660           0.996              0.336        0.228         0.435     0.753
                 deu

**L'explication de Moussa, à vérifier :** dans `prompts.md` (prompt 3), Moussa explique le faible gain en élimination directe par des matchs **plus serrés** : à écart d'Elo égal (moins de 150 points), le gain serait le même dans les deux phases. On le vérifie, avec les intervalles à 95 %.

In [7]:
lignes = []
for serre, nom_serre in [(True, 'écart < 150 points'), (False, 'écart ≥ 150 points')]:
    for phase in ['groupes / phase de ligue', 'élimination directe']:
        d = t[((t['ecart_elo_complet'].abs() < 150) == serre) & (t['phase'] == phase)]
        gain, bas, haut = ic_ecart(d['perte_dummy'].to_numpy(), d['perte_poisson'].to_numpy())
        lignes.append({'écart d\'Elo': nom_serre, 'phase': phase, 'matchs': len(d),
                       'écart moyen': d['ecart_elo_complet'].abs().mean(), 'gain sur le Dummy': gain,
                       'IC 95 % bas': bas, 'IC 95 % haut': haut})
print(pd.DataFrame(lignes).set_index(["écart d'Elo", 'phase']).round(3).to_string())

                                             matchs  écart moyen  gain sur le Dummy  IC 95 % bas  IC 95 % haut
écart d'Elo        phase                                                                                      
écart < 150 points groupes / phase de ligue     218       68.294              0.029       -0.004         0.060
                   élimination directe           78       61.159              0.027       -0.025         0.076
écart ≥ 150 points groupes / phase de ligue     166      254.422              0.292        0.201         0.376
                   élimination directe           40      213.456              0.048       -0.168         0.240


**Lecture :**
- **Pour les matchs serrés, l'explication de Moussa est confirmée** : à moins de 150 points d'écart, le gain est le même en phase de groupes (**+0,029**) et en élimination directe (**+0,027**). L'élimination directe a simplement plus de matchs serrés (78 sur 118).
- **Pour les matchs déséquilibrés, elle reste à prouver** : à 150 points ou plus, le gain est de **+0,292** en phase de groupes, mais seulement de **+0,048** en élimination directe. L'écart est grand, mais il ne repose que sur **40 matchs**, avec un intervalle de −0,168 à +0,240 : on ne peut rien conclure.
- **Pour la simulation, le constat ne change pas** : en élimination directe, le gain sur le Dummy n'est démontré dans aucun des deux groupes.

**Un résultat suspect à vérifier tout de suite :** le modèle est **le meilleur** sur les matchs où au moins un Elo est **estimé** (log loss 0,660). Un auditeur doit se méfier : une estimation qui « devine trop bien » peut cacher une fuite. Mais il y a une explication plus simple : les clubs à Elo estimé sont surtout de petits clubs (Ukraine, Croatie, Suisse…) qui affrontent de grands clubs, donc des matchs **très déséquilibrés**, faciles à prédire. On compare donc **à écart d'Elo égal**.

In [8]:
t['origine'] = decoupages["origine de l'Elo"]
t['tranche'] = pd.cut(t['ecart_elo_complet'].abs(), [0, 100, 200, 300, 2000], right=False,
                      labels=['0-99', '100-199', '200-299', '300 et plus'])
controle = t.groupby(['tranche', 'origine'], observed=True).agg(
    matchs=('game_id', 'size'), log_loss_poisson=('perte_poisson', 'mean'), log_loss_dummy=('perte_dummy', 'mean'))
print("Écart d'Elo moyen (valeur absolue) :", t.groupby('origine')['ecart_elo_complet'].apply(lambda e: e.abs().mean()).round(0).to_dict())
print(controle.round(3).to_string())

Écart d'Elo moyen (valeur absolue) : {'au moins un Elo estimé': 235.0, 'deux Elo officiels': 119.0}
                                    matchs  log_loss_poisson  log_loss_dummy
tranche     origine                                                         
0-99        au moins un Elo estimé      18             1.077           1.130
            deux Elo officiels         197             1.022           1.043
100-199     au moins un Elo estimé      23             0.783           0.978
            deux Elo officiels         135             0.914           0.992
200-299     au moins un Elo estimé      20             0.590           0.952
            deux Elo officiels          62             0.876           1.054
300 et plus au moins un Elo estimé      32             0.380           0.960
            deux Elo officiels          15             0.840           0.963


**Lecture :**
- **Les matchs à Elo estimé sont bien plus déséquilibrés** : écart d'Elo moyen de **235 points**, contre 119 quand les deux Elo sont officiels. Dans 98 % de ces matchs, le club à Elo estimé est le **plus faible** des deux, et le favori gagne **75 %** du temps.
- **Mais cela n'explique pas tout : à écart d'Elo égal, ils restent mieux prédits.** Entre 200 et 299 points d'écart : 0,590 contre 0,876. À 300 points et plus : 0,380 contre 0,840.
- **Deux explications possibles, qu'on ne peut pas départager :**
  1. l'Elo officiel **resserre** les écarts entre grands et petits championnats, et l'estimation « par les résultats » les mesure mieux. Ce serait une qualité de l'Elo complet ;
  2. l'estimation contient **un peu d'information du futur** (le recalage du test 5), qui rendrait ces matchs « trop » faciles.
- **Les groupes sont petits** (15 à 32 matchs par case) : un auditeur ne peut ni confirmer une fuite, ni l'écarter. **On classe ce point comme suspect, non résolu.**
- **Désaccord avec l'explication de Moussa** (`prompts.md`, prompt 3), qui conclut que « l'estimation ne dégrade pas les prédictions » parce que ces matchs sont déséquilibrés. Le déséquilibre n'explique qu'une partie de l'écart : la question reste ouverte.

**Lecture :**

| Sous-groupe (test) | Matchs | Log loss Poisson | Gain sur le Dummy (IC 95 %) | Accuracy |
|---|---|---|---|---|
| Groupes / phase de ligue | 384 | 0,873 | **+0,143** (+0,099 à +0,185) | 60,4 % |
| **Élimination directe** | **118** | 0,999 | **+0,034 (−0,043 à +0,106)** | 52,5 % |
| Grand écart d'Elo (≥ 100) | 287 | 0,809 | +0,187 (+0,122 à +0,248) | 67,2 % |
| Petit écart d'Elo (< 100) | 215 | 1,026 | +0,024 (+0,002 à +0,046) | 47,0 % |
| Deux Elo officiels | 409 | 0,957 | +0,068 (+0,027 à +0,106) | 54,8 % |
| Au moins un Elo estimé | 93 | 0,660 | +0,336 (+0,228 à +0,435) | 75,3 % |

- **Le résultat le plus important de l'audit : en élimination directe, on ne peut pas dire que le modèle bat le Dummy.** Le gain est de +0,034, et l'intervalle **contient 0** (118 matchs). Or c'est là que se décide le titre : la simulation repose, pour ses tours les plus importants, sur un modèle qui n'est pas démontré meilleur que les fréquences passées.
- **Le modèle brille surtout sur les matchs faciles** : grand écart d'Elo (gain +0,187) contre petit écart (+0,024, à peine au-dessus de 0). Les matchs serrés, ceux qui comptent le plus pour le classement, sont presque aussi difficiles pour lui que pour le Dummy.
- **La moyenne générale (0,902) cache donc ces écarts** : elle est tirée par la phase de groupes et par les matchs déséquilibrés.

## 6. Test 5 : la fuite de l'Elo « maison recalé »

**Le problème repéré dans `v2_elo.ipynb` (section 5.2) :** quand un club n'a ni Elo officiel ni assez de matchs contre des clubs connus, son Elo complet est son Elo maison **recalé** sur l'échelle officielle avec une droite. Mais cette droite est ajustée **sur toute la saison** (`np.polyfit` par saison), donc aussi sur des matchs joués **après** le match prédit. C'est une fuite, au moins en principe.

**Le test :** combien de matchs sont concernés ? Et le modèle change-t-il si on les retire, de l'apprentissage comme de l'évaluation ?

In [9]:
recale = g['home_source_elo'].str.contains('recalé') | g['away_source_elo'].str.contains('recalé')
lignes_recalees = g['home_source_elo'].str.contains('recalé').sum() + g['away_source_elo'].str.contains('recalé').sum()
print(f"Lignes club-match à Elo recalé : {lignes_recalees} sur {2 * len(g):,} ({lignes_recalees / (2 * len(g)):.1%})")
print(f"Matchs concernés : {recale.sum()} sur {len(g):,} ({recale.mean():.1%}), dont {recale[g['season'] >= 2023].sum()} dans le test")
print(g[recale].groupby('season').size().rename('matchs').to_frame().T.to_string())

propre = g[~recale]
appr_p, test_p = propre[propre['season'] <= 2022], propre[propre['season'] >= 2023]
# Pour comparer honnêtement, les deux modèles sont jugés sur les MÊMES matchs : ceux sans Elo recalé
comparaison = pd.DataFrame({
    'appris sur tous les matchs': {
        'matchs appris (final)': len(g), 'coef Elo (final)': entrainer(g, VARIABLES)['ecart_100'],
        'coef terrain (final)': entrainer(g, VARIABLES)['terrain'],
        'log loss glissante': moyenne_ponderee(glissante(VARIABLES, g, propre), 'log loss'),
        'log loss test': perte(test_p['resultat'], predire(appr_test, test_p, VARIABLES)).mean()},
    'appris sans Elo recalé': {
        'matchs appris (final)': len(propre), 'coef Elo (final)': entrainer(propre, VARIABLES)['ecart_100'],
        'coef terrain (final)': entrainer(propre, VARIABLES)['terrain'],
        'log loss glissante': moyenne_ponderee(glissante(VARIABLES, propre, propre), 'log loss'),
        'log loss test': perte(test_p['resultat'], predire(appr_p, test_p, VARIABLES)).mean()},
}).T
print(f"\nLes deux modèles sont jugés sur les mêmes matchs, sans Elo recalé "
      f"({propre['season'].between(2015, 2022).sum()} en validation, {len(test_p)} en test) :")
print(comparaison.round(4).to_string())

# Les matchs recalés eux-mêmes : le modèle y est-il anormalement bon ?
r = g[recale & g['season'].between(2015, 2022)]
v_r = glissante(VARIABLES, g, r)
print(f"\nSur les {len(r)} matchs recalés de la validation : log loss Poisson {moyenne_ponderee(v_r, 'log loss'):.4f}"
      f" | Dummy {moyenne_ponderee(v_r, 'log loss Dummy'):.4f} | écart d'Elo moyen (en valeur absolue) {r['ecart_elo_complet'].abs().mean():.0f}")

Lignes club-match à Elo recalé : 81 sur 3,742 (2.2%)
Matchs concernés : 79 sur 1,871 (4.2%), dont 10 dans le test
season  2012  2014  2015  2016  2017  2018  2020  2021  2022  2023  2025
matchs    20     6    12     6     6     3     4     6     6     1     9



Les deux modèles sont jugés sur les mêmes matchs, sans Elo recalé (951 en validation, 492 en test) :
                            matchs appris (final)  coef Elo (final)  coef terrain (final)  log loss glissante  log loss test
appris sur tous les matchs                 1871.0            0.1989                0.1284              0.9207          0.905
appris sans Elo recalé                     1792.0            0.1975                0.1314              0.9210          0.905

Sur les 43 matchs recalés de la validation : log loss Poisson 0.7861 | Dummy 1.0854 | écart d'Elo moyen (en valeur absolue) 281


**Lecture :**
- **L'ordre de grandeur annoncé (≈ 2 %) est juste en lignes, pas en matchs** : **81 lignes club-match sur 3 742 (2,2 %)**, mais **79 matchs sur 1 871 (4,2 %)**, car une seule équipe recalée suffit à concerner le match. 10 de ces matchs sont dans le test.
- **Le modèle ne change pas quand on les retire** : coefficient de l'Elo **0,1989 contre 0,1975**, terrain **0,128 contre 0,131**. Jugés sur les mêmes matchs, les deux modèles ont la même log loss : **0,9207 contre 0,9210** en validation, **0,905** dans les deux cas sur le test.
- **Sur les 43 matchs recalés de la validation, le modèle est très bon** (0,786 contre 1,085 pour le Dummy). Ce sont des matchs très déséquilibrés (281 points d'écart en moyenne), donc ce n'est pas une preuve de fuite.
- **Verdict :** la fuite existe **en principe** (la droite de recalage utilise des matchs futurs), mais son **effet mesurable est nul**. Correction conseillée à Moussa, dans `v2_elo` : ajuster la droite sur les **saisons précédentes** seulement.

## 7. Audit du test de la forme récente

**Ce qui est audité :** le test de la forme de `MODEL_POISSON.ipynb`. On ne refait pas la décision : on **recalcule de notre côté** pour vérifier qu'on trouve pareil, avec la **règle fixée à l'avance** par le groupe :

> la forme est gardée **seulement si** la log loss moyenne de la validation glissante baisse **ET** si elle fait mieux sur **au moins 5 saisons sur 8**.

Le test 2023-2025 n'est **pas** utilisé ici.

Trois variantes :
1. **tous les matchs**, comme dans `MODEL_POISSON` : les 19 écarts de forme manquants sont remplacés par la médiane de l'apprentissage. Comme chaque match y apparaît deux fois, avec des écarts opposés (+x pour une équipe, −x pour l'autre), cette médiane vaut exactement 0 : c'est « forme égale » ;
2. **sans les 19 matchs** à forme manquante, pour voir si ce choix change quelque chose ;
3. **sans les clubs dont le championnat manque dans `games.csv`** (on garde seulement les matchs où les deux championnats sont couverts) : pour eux, la « forme » est faite surtout de matchs de coupe et d'Europe.

In [10]:
def audit_forme(donnees):
    sans = glissante(VARIABLES, donnees)
    avec = glissante(VARIABLES + ['ecart_forme'], donnees)
    mieux = (avec['log loss'] < sans['log loss']).sum()
    baisse = moyenne_ponderee(avec, 'log loss') - moyenne_ponderee(sans, 'log loss')
    return {'matchs (validation)': int(sans['matchs'].sum()), 'log loss sans forme': moyenne_ponderee(sans, 'log loss'),
            'log loss avec forme': moyenne_ponderee(avec, 'log loss'), 'écart (avec − sans)': baisse,
            'saisons où la forme fait mieux': f"{mieux} / {len(sans)}",
            'décision': 'gardée' if (baisse < 0 and mieux >= 5) else 'rejetée'}, sans, avec

variantes = {'1. tous les matchs (comme MODEL_POISSON)': g,
             '2. sans les 19 matchs à forme manquante': g[g['ecart_forme_5'].notna()],
             '3. les deux championnats couverts': g[g['deux_couverts']]}
resultats_forme, detail_saisons = {}, {}
for nom, donnees in variantes.items():
    resultats_forme[nom], sans, avec = audit_forme(donnees)
    detail_saisons[nom] = (avec['log loss'] - sans['log loss'])
resultats_forme = pd.DataFrame(resultats_forme).T
print(resultats_forme.to_string())

# Vérification : on retrouve les chiffres de la section 8 de MODEL_POISSON (sans 0,9148, avec 0,9156, 4 saisons sur 8)
v1 = resultats_forme.iloc[0]
assert round(v1['log loss sans forme'], 4) == 0.9148 and round(v1['log loss avec forme'], 4) == 0.9156
assert v1['saisons où la forme fait mieux'] == '4 / 8' and v1['décision'] == 'rejetée'
print("\nChiffres de MODEL_POISSON (section 8) retrouvés : même décision, « rejetée ».")

print("\nÉcart de log loss par saison (avec − sans forme ; négatif = la forme aide) :")
print(pd.DataFrame(detail_saisons).round(4).T.to_string())

# Coefficient de la forme, appris sur 2012-2022
c = entrainer(g[g['season'] <= 2022], VARIABLES + ['ecart_forme'])
print(f"\nCoefficient de la forme (2012-2022) : {c['ecart_forme']:+.4f} → +1 point de forme d'écart = buts × {np.exp(c['ecart_forme']):.3f}")

                                         matchs (validation) log loss sans forme log loss avec forme écart (avec − sans) saisons où la forme fait mieux décision
1. tous les matchs (comme MODEL_POISSON)                 994            0.914843            0.915594            0.000751                          4 / 8  rejetée
2. sans les 19 matchs à forme manquante                  983            0.915207            0.915952            0.000745                          4 / 8  rejetée
3. les deux championnats couverts                        814            0.939172            0.939748            0.000576                          3 / 8  rejetée

Chiffres de MODEL_POISSON (section 8) retrouvés : même décision, « rejetée ».

Écart de log loss par saison (avec − sans forme ; négatif = la forme aide) :
saison                                      2015    2016    2017    2018    2019    2020    2021    2022
1. tous les matchs (comme MODEL_POISSON) -0.0014 -0.0017  0.0053 -0.0008  0.0043  0.0012  0.00

**Lecture :**

| Variante (validation glissante 2015-2022) | Matchs | Sans forme | Avec forme | Saisons où la forme aide | Décision |
|---|---|---|---|---|---|
| 1. Tous les matchs (comme MODEL_POISSON) | 994 | 0,9148 | 0,9156 | 4 / 8 | rejetée |
| 2. Sans les 19 matchs à forme manquante | 983 | 0,9152 | 0,9160 | 4 / 8 | rejetée |
| 3. Les deux championnats couverts | 814 | 0,9392 | 0,9397 | 3 / 8 | rejetée |

- **On trouve exactement les mêmes chiffres que `MODEL_POISSON`** (0,9148 contre 0,9156, 4 saisons sur 8). Les `assert` le vérifient. La décision « **rejetée** » est donc reproduite de façon indépendante.
- **La règle a été respectée** : la log loss moyenne **monte** (+0,0008), et la forme n'aide que sur 4 saisons sur 8. Il fallait les deux conditions, aucune n'est remplie.
- **La décision ne change pas** quand on retire les 19 matchs à forme manquante, ni quand on garde seulement les clubs dont le championnat est couvert (la forme fait même un peu moins bien : 3 saisons sur 8).
- **Le coefficient de la forme est minuscule** : +1 point de forme d'écart = buts × **1,033**. C'est cohérent avec `v2_forme.ipynb` : une fois l'Elo pris en compte, la forme n'ajoute presque rien (corrélation partielle +0,035).
- **La règle était-elle vraiment fixée à l'avance ?** Oui, en partie : elle est écrite dans le commit `65741bd`, exécuté par Moussa alors que `forme_par_match.csv` n'existait pas encore chez lui (le notebook commité affiche « section sautée »). Mais `MODEL_POISSON` mentionne aussi un **calcul rapide fait hors du dépôt** avant ce commit, qui donnait déjà « 4 saisons sur 8 ». La règle n'a donc pas été écrite totalement à l'aveugle. Ce n'est pas grave ici : la règle mène au **rejet**, le choix le plus prudent, qui n'embellit pas le modèle.

## 8. Conclusion : ce qui résiste, ce qui casse, ce qu'on ne peut pas affirmer

### ✅ Ce qui résiste
- **Pas de fuite par la cible** : avec des scores mélangés, le modèle tombe au niveau « terrain seul » (1,026 contre 0,902).
- **L'Elo officiel est bien connu avant le match** : 100 % des 3 348 valeurs existent à une date antérieure (7 jours en général).
- **L'Elo et le terrain apportent un gain réel** : les retirer dégrade le test de +0,126 et +0,022, et les intervalles ne contiennent pas 0.
- **La fuite du recalage n'a aucun effet mesurable** : retirer les 79 matchs concernés ne change ni les coefficients ni la log loss.
- **Le rejet de la forme récente est solide** : même décision dans les 3 variantes, et chiffres de `MODEL_POISSON` reproduits à l'identique.

### ❌ Ce qui casse ou reste fragile
- **En élimination directe (118 matchs), le modèle n'est pas démontré meilleur que le Dummy** : gain de +0,034, intervalle de −0,043 à +0,106. C'est pourtant là que se joue le titre.
- **Sur les matchs serrés (écart d'Elo < 100), le gain est à peine au-dessus de 0** (+0,024).
- **Le nouveau format n'est pas démontré** : +0,0016 sur 188 matchs, intervalle qui contient 0. Et l'idée comme sa validation viennent du **test**, qui n'est donc plus vierge.

### ❓ Ce qu'on ne peut pas affirmer
- **Que l'Elo estimé est sans fuite** : à écart égal, les matchs à Elo estimé sont bien mieux prédits que les autres (0,380 contre 0,840 au-delà de 300 points), sur de petits groupes. Suspect, non résolu.
- **Que le score de 0,902 se retrouvera en 2026-2027** : le test a été réutilisé (sections 6-7 de `MODEL_POISSON` et cet audit), et l'Elo de 2026-2027 est figé au 8 septembre.
- **Qu'aucune forme récente ne peut aider** : on a testé une seule définition (5 matchs, 60 jours, sans tenir compte de la force des adversaires).

### Recommandations pour l'équipe (à décider par le groupe)
1. **Dire dans la soutenance** que le modèle est fiable sur la phase de ligue, et **incertain en élimination directe**. Les probabilités de titre de la simulation héritent de cette incertitude.
2. **Présenter le nouveau format comme une hypothèse**, pas comme un gain prouvé, et signaler que le test 2023-2025 a servi à le choisir.
3. **Dans `v2_elo`**, ajuster la droite de recalage sur les saisons précédentes seulement. L'effet est nul aujourd'hui, mais la règle « aucune fuite » serait alors respectée à 100 %.